In [ ]:

# ===============================================
# In This file:
# The Multi-generator R-GAN Framework is implemented on the CIFAR-10 dataset for targeted attacks. 
# This code considers the worst-case scenario for CIFAR-10, where 10 generators are used, according tho the R-GAN paper.
# Each generator forces perturbed data towards a specific target class.
# The accuracy of the R-net, C-net, and T-net classifiers is evaluated against attacks generated by each generator.
# ===============================================


In [13]:
# === CHANGE THIS PATH to your actual Desktop path ===
dataset_path = r"C:\Users\Administrator\Desktop\R-GAN\Dataset\cifar-10-python"

In [ ]:

# ===============================================
# In This Cell: The CIFAR-10 dataset is loaded.
# ===============================================


import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define transforms 
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])




train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=True,
    transform=transform_train,
    download=False   
)

test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=False,
    transform=transform_test,
    download=False
)

batch_size = 64
num_workers = 2 if torch.cuda.is_available() else 0

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("CIFAR-10 loaded successfully!")
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


In [ ]:

# ===============================================
# In This Cell: The data is normalized.
# ===============================================


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

def denormalize(imgs_norm):
    """imgs_norm: normalized (mean/std) -> returns imgs in [0,1] pixel space"""
    return imgs_norm * CIFAR10_STD + CIFAR10_MEAN

def renormalize(imgs_01):
    """imgs_01 in [0,1] -> normalized"""
    return (imgs_01 - CIFAR10_MEAN) / CIFAR10_STD

# L_inf epsilon in pixel space
eps = 8.0 / 255.0
print("eps (L_inf):", eps)


In [ ]:

# ===============================================
# In This Cell:
# The architecture of the classifiers f-net (C-net) and R-net is defined as ResNet-18.
# The pretrained classifier C-net described in the paper is referred to as f-net in the code (f-net in code = C-net in the paper).
# ===============================================

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ResNet-18 adapted for CIFAR-10 
class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_CIFAR, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)  # global avg pool
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18_CIFAR():
    return ResNet_CIFAR(BasicBlock, [2,2,2,2])


In [17]:
# ===============================================
# In This Cell:
# The architecture of generators is defined.
# ===============================================

class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.BatchNorm2d, use_dropout=False, use_bias=False):
        super().__init__()
        conv_block = [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim),
            nn.ReLU(True)
        ]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]
        conv_block += [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim)
        ]
        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        return x + self.conv_block(x)

class Generator(nn.Module):
    def __init__(self, gen_input_nc=3, image_nc=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(gen_input_nc, 32, 3, 1, 1), nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, 2, 1),           nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, 2, 1),          nn.InstanceNorm2d(128), nn.ReLU(True),
        )
        self.bottleneck = nn.Sequential(ResnetBlock(128), ResnetBlock(128), ResnetBlock(128))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),  nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, image_nc, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, x_01):
        # x_01 in [0,1]
        return self.decoder(self.bottleneck(self.encoder(x_01)))


In [18]:
# ===============================================
# In This Cell:
# The classifier f-net (C-net) is trained using ResNet-18.
# ===============================================


f_net = ResNet18_CIFAR().to(device)
criterion = nn.CrossEntropyLoss()
optimizer_f = optim.SGD(f_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_f = optim.lr_scheduler.MultiStepLR(optimizer_f, milestones=[80, 120], gamma=0.1)

epochs_f = 150
print("Training f_net (ResNet18) for", epochs_f, "epochs...")

for epoch in range(1, epochs_f+1):
    # Train
    f_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm, labels = imgs_norm.to(device), labels.to(device)
        optimizer_f.zero_grad()
        logits = f_net(imgs_norm)  # imgs_norm already normalized by transform_train
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_f.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total
    train_loss = running_loss / len(train_dataloader)

    # Test
    if (epoch ==1 or epoch % 10 == 0 or epoch==epochs_f):

        f_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm, labels_t = imgs_t_norm.to(device), labels_t.to(device)
                logits_t = f_net(imgs_t_norm)
                loss_t = criterion(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)
        test_acc = 100.0 * correct_test / total_test
        test_loss = test_loss / len(test_dataloader)
        print(f"[f_train] Epoch {epoch}/{epochs_f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")
    scheduler_f.step()

# Freeze f_net and set eval
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False


In [ ]:

# ===============================================
# In This Cell:
# The Multi(Ten)-generator R-GAN Framework is trained.
# R-net and Generators(G_i) are trained together competitively in the R-GAN Framework according to the R-GAN paper describtion.
# Each generator trains to produce a specific target attack (targets 0..9).
# Generator G_i forces both f-net & R-net classifiers towards the class with label "i" (e.g., target G0 = class 0) .
# ===============================================

import torch
import torch.nn.functional as F
import torch.optim as optim
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Build R_net and ten generators ---
R_net = ResNet18_CIFAR().to(device)
optimizer_R = optim.SGD(R_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_R = optim.lr_scheduler.MultiStepLR(optimizer_R, milestones=[80,120], gamma=0.1)

G0 = Generator(gen_input_nc=3, image_nc=3).to(device)
G1 = Generator(gen_input_nc=3, image_nc=3).to(device)
G2 = Generator(gen_input_nc=3, image_nc=3).to(device)
G3 = Generator(gen_input_nc=3, image_nc=3).to(device)
G4 = Generator(gen_input_nc=3, image_nc=3).to(device)
G5 = Generator(gen_input_nc=3, image_nc=3).to(device)
G6 = Generator(gen_input_nc=3, image_nc=3).to(device)
G7 = Generator(gen_input_nc=3, image_nc=3).to(device)
G8 = Generator(gen_input_nc=3, image_nc=3).to(device)
G9 = Generator(gen_input_nc=3, image_nc=3).to(device)

optG0 = torch.optim.Adam(G0.parameters(), lr=1e-3)
optG1 = torch.optim.Adam(G1.parameters(), lr=1e-3)
optG2 = torch.optim.Adam(G2.parameters(), lr=1e-3)
optG3 = torch.optim.Adam(G3.parameters(), lr=1e-3)
optG4 = torch.optim.Adam(G4.parameters(), lr=1e-3)
optG5 = torch.optim.Adam(G5.parameters(), lr=1e-3)
optG6 = torch.optim.Adam(G6.parameters(), lr=1e-3)
optG7 = torch.optim.Adam(G7.parameters(), lr=1e-3)
optG8 = torch.optim.Adam(G8.parameters(), lr=1e-3)
optG9 = torch.optim.Adam(G9.parameters(), lr=1e-3)

# Freeze pretrained f_net 
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False

# Hyperparams 
eps = 8.0/255.0                   # L_inf pixel-space
lambda_f = 5.0
lambda_R = 5.0
lambda_pert = 1.0
num_epochs = 150                 # change as needed
print_every = 1
grad_clip_max = 5.0

# Targets each generator pushes to
target0=0; target1=1; target2=2; target3=3; target4=4
target5=5; target6=6; target7=7; target8=8; target9=9

def compute_target_loss_and_pert(G, imgs_pixel, target_label, device):
    """Returns (loss_f_target, loss_R_target, loss_pert, adv_norm, delta)"""
    delta = G(imgs_pixel)
    delta = torch.clamp(delta, -eps, eps)
    adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
    adv_norm = renormalize(adv_pixel)
    target_labels = torch.full_like(target_label, fill_value=target_label[0].item() if target_label.numel()>0 else target_label, device=device)
    # Note: caller will pass appropriate target_labels; here we accept prebuilt target_labels
    return delta, adv_norm

print("Starting training: generators-first, then aggregated R-update per minibatch.")

for epoch in range(1, num_epochs+1):
    t0 = time.time()
    R_net.train()
    G0.train(); G1.train(); G2.train(); G3.train(); G4.train()
    G5.train(); G6.train(); G7.train(); G8.train(); G9.train()

    # accumulators for printing
    total_loss_R = 0.0
    total_loss_G = [0.0]*10
    total_loss_f = [0.0]*10
    total_loss_pert = [0.0]*10
    batches = 0

    for imgs_norm, labels in train_dataloader:
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)
        batches += 1

        imgs_pixel = denormalize(imgs_norm)  # pixel-space

        # ---------------------------
        # (A) Update generators sequentially (R frozen)
        # ---------------------------
        R_net.eval()
        f_net.eval()
        for p in R_net.parameters(): p.requires_grad = False

        adv_norms = []          # store adv_norm (detached) for R aggregated update
        adv_losses_for_R = []   # store per-generator adv loss (cross-entropy w.r.t. true labels) 

        # helper to process one G block to return required items
        def G_block(G, optG, target_label_int, idx):
            # produce delta and adv_norm
            delta = G(imgs_pixel)
            delta = torch.clamp(delta, -eps, eps)
            adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
            adv_norm = renormalize(adv_pixel)

            # targeted losses
            target_labels = torch.full_like(labels, fill_value=target_label_int, device=device)
            logits_f = f_net(adv_norm)
            loss_f_target = F.cross_entropy(logits_f, target_labels)
            logits_R = R_net(adv_norm)
            loss_R_target = F.cross_entropy(logits_R, target_labels)

            loss_pert = torch.mean(torch.norm(delta.view(delta.size(0), -1), dim=1, p=2))
            loss_G = lambda_f * loss_f_target + lambda_R * loss_R_target + lambda_pert * loss_pert

            optG.zero_grad()
            loss_G.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), grad_clip_max)
            optG.step()

            return loss_G.item(), loss_f_target.item(), loss_pert.item(), adv_norm.detach(), F.cross_entropy(logits_R, labels).item()

        # run each generator block explicitly in desired order
        out0 = G_block(G0, optG0, target0, 0); total_loss_G[0]+=out0[0]; total_loss_f[0]+=out0[1]; total_loss_pert[0]+=out0[2]; adv_norms.append(out0[3]); adv_losses_for_R.append(out0[4])
        out1 = G_block(G1, optG1, target1, 1); total_loss_G[1]+=out1[0]; total_loss_f[1]+=out1[1]; total_loss_pert[1]+=out1[2]; adv_norms.append(out1[3]); adv_losses_for_R.append(out1[4])
        out2 = G_block(G2, optG2, target2, 2); total_loss_G[2]+=out2[0]; total_loss_f[2]+=out2[1]; total_loss_pert[2]+=out2[2]; adv_norms.append(out2[3]); adv_losses_for_R.append(out2[4])
        out3 = G_block(G3, optG3, target3, 3); total_loss_G[3]+=out3[0]; total_loss_f[3]+=out3[1]; total_loss_pert[3]+=out3[2]; adv_norms.append(out3[3]); adv_losses_for_R.append(out3[4])
        out4 = G_block(G4, optG4, target4, 4); total_loss_G[4]+=out4[0]; total_loss_f[4]+=out4[1]; total_loss_pert[4]+=out4[2]; adv_norms.append(out4[3]); adv_losses_for_R.append(out4[4])
        out5 = G_block(G5, optG5, target5, 5); total_loss_G[5]+=out5[0]; total_loss_f[5]+=out5[1]; total_loss_pert[5]+=out5[2]; adv_norms.append(out5[3]); adv_losses_for_R.append(out5[4])
        out6 = G_block(G6, optG6, target6, 6); total_loss_G[6]+=out6[0]; total_loss_f[6]+=out6[1]; total_loss_pert[6]+=out6[2]; adv_norms.append(out6[3]); adv_losses_for_R.append(out6[4])
        out7 = G_block(G7, optG7, target7, 7); total_loss_G[7]+=out7[0]; total_loss_f[7]+=out7[1]; total_loss_pert[7]+=out7[2]; adv_norms.append(out7[3]); adv_losses_for_R.append(out7[4])
        out8 = G_block(G8, optG8, target8, 8); total_loss_G[8]+=out8[0]; total_loss_f[8]+=out8[1]; total_loss_pert[8]+=out8[2]; adv_norms.append(out8[3]); adv_losses_for_R.append(out8[4])
        out9 = G_block(G9, optG9, target9, 9); total_loss_G[9]+=out9[0]; total_loss_f[9]+=out9[1]; total_loss_pert[9]+=out9[2]; adv_norms.append(out9[3]); adv_losses_for_R.append(out9[4])

        # Unfreeze R for its aggregated update
        for p in R_net.parameters(): p.requires_grad = True

        # ---------------------------
        # (B) Aggregated R update (single update using clean + average adv loss)
        # ---------------------------
        R_net.train()
        optimizer_R.zero_grad()

        logits_clean = R_net(imgs_norm)
        loss_R_clean = F.cross_entropy(logits_clean, labels)

        # average adv loss across adv_norms (len = 10)
        loss_R_adv_sum = 0.0
        for adv_norm_t in adv_norms:
            logits_R_adv_t = R_net(adv_norm_t)
            loss_R_adv_sum = loss_R_adv_sum + F.cross_entropy(logits_R_adv_t, labels)
        loss_R_adv = loss_R_adv_sum / float(len(adv_norms))

        loss_R = loss_R_clean + loss_R_adv
        loss_R.backward()
        torch.nn.utils.clip_grad_norm_(R_net.parameters(), grad_clip_max)
        optimizer_R.step()

        total_loss_R += loss_R.item()

    # --- end minibatch loop ---

    # compute epoch averages
    denom = (batches if batches>0 else 1)
    avg_R = total_loss_R / denom
    avg_Gs = [total_loss_G[i]/denom for i in range(10)]
    avg_fs = [total_loss_f[i]/denom for i in range(10)]
    avg_perts = [total_loss_pert[i]/denom for i in range(10)]

    if epoch % print_every == 0:
        print(f"[Epoch {epoch}/{num_epochs}] Loss_R(avg): {avg_R:.4f} | Loss_Gs(avg): {', '.join(f'{x:.4f}' for x in avg_Gs)}")
        print("    Loss_f_targets (per G):", ", ".join(f"{x:.4f}" for x in avg_fs))
        print("    Pert (L2) per G:", ", ".join(f"{x:.4f}" for x in avg_perts))

    # ----- Evaluation on test set -----
    G0.eval(); G1.eval(); G2.eval(); G3.eval(); G4.eval()
    G5.eval(); G6.eval(); G7.eval(); G8.eval(); G9.eval()
    R_net.eval(); f_net.eval()

    total_test = 0
    correct_clean_R = correct_clean_f = 0

    acc_adv_R_correct = [0]*10
    acc_adv_f_correct = [0]*10

    with torch.no_grad():
        for imgs_t_norm, labels_t in test_dataloader:
            imgs_t_norm = imgs_t_norm.to(device)
            labels_t = labels_t.to(device)
            batch_n = labels_t.size(0)
            total_test += batch_n

            preds_R_clean = R_net(imgs_t_norm).argmax(dim=1)
            preds_f_clean = f_net(imgs_t_norm).argmax(dim=1)
            correct_clean_R += (preds_R_clean == labels_t).sum().item()
            correct_clean_f += (preds_f_clean == labels_t).sum().item()

            imgs_t_pixel = denormalize(imgs_t_norm)

            # advs by each generator
            d0 = torch.clamp(G0(imgs_t_pixel), -eps, eps); adv0_norm = renormalize(torch.clamp(imgs_t_pixel + d0, 0.0, 1.0))
            d1 = torch.clamp(G1(imgs_t_pixel), -eps, eps); adv1_norm = renormalize(torch.clamp(imgs_t_pixel + d1, 0.0, 1.0))
            d2 = torch.clamp(G2(imgs_t_pixel), -eps, eps); adv2_norm = renormalize(torch.clamp(imgs_t_pixel + d2, 0.0, 1.0))
            d3 = torch.clamp(G3(imgs_t_pixel), -eps, eps); adv3_norm = renormalize(torch.clamp(imgs_t_pixel + d3, 0.0, 1.0))
            d4 = torch.clamp(G4(imgs_t_pixel), -eps, eps); adv4_norm = renormalize(torch.clamp(imgs_t_pixel + d4, 0.0, 1.0))
            d5 = torch.clamp(G5(imgs_t_pixel), -eps, eps); adv5_norm = renormalize(torch.clamp(imgs_t_pixel + d5, 0.0, 1.0))
            d6 = torch.clamp(G6(imgs_t_pixel), -eps, eps); adv6_norm = renormalize(torch.clamp(imgs_t_pixel + d6, 0.0, 1.0))
            d7 = torch.clamp(G7(imgs_t_pixel), -eps, eps); adv7_norm = renormalize(torch.clamp(imgs_t_pixel + d7, 0.0, 1.0))
            d8 = torch.clamp(G8(imgs_t_pixel), -eps, eps); adv8_norm = renormalize(torch.clamp(imgs_t_pixel + d8, 0.0, 1.0))
            d9 = torch.clamp(G9(imgs_t_pixel), -eps, eps); adv9_norm = renormalize(torch.clamp(imgs_t_pixel + d9, 0.0, 1.0))

            preds_R_adv0 = R_net(adv0_norm).argmax(dim=1); preds_f_adv0 = f_net(adv0_norm).argmax(dim=1)
            preds_R_adv1 = R_net(adv1_norm).argmax(dim=1); preds_f_adv1 = f_net(adv1_norm).argmax(dim=1)
            preds_R_adv2 = R_net(adv2_norm).argmax(dim=1); preds_f_adv2 = f_net(adv2_norm).argmax(dim=1)
            preds_R_adv3 = R_net(adv3_norm).argmax(dim=1); preds_f_adv3 = f_net(adv3_norm).argmax(dim=1)
            preds_R_adv4 = R_net(adv4_norm).argmax(dim=1); preds_f_adv4 = f_net(adv4_norm).argmax(dim=1)
            preds_R_adv5 = R_net(adv5_norm).argmax(dim=1); preds_f_adv5 = f_net(adv5_norm).argmax(dim=1)
            preds_R_adv6 = R_net(adv6_norm).argmax(dim=1); preds_f_adv6 = f_net(adv6_norm).argmax(dim=1)
            preds_R_adv7 = R_net(adv7_norm).argmax(dim=1); preds_f_adv7 = f_net(adv7_norm).argmax(dim=1)
            preds_R_adv8 = R_net(adv8_norm).argmax(dim=1); preds_f_adv8 = f_net(adv8_norm).argmax(dim=1)
            preds_R_adv9 = R_net(adv9_norm).argmax(dim=1); preds_f_adv9 = f_net(adv9_norm).argmax(dim=1)

            acc_adv_R_correct[0] += (preds_R_adv0 == labels_t).sum().item()
            acc_adv_R_correct[1] += (preds_R_adv1 == labels_t).sum().item()
            acc_adv_R_correct[2] += (preds_R_adv2 == labels_t).sum().item()
            acc_adv_R_correct[3] += (preds_R_adv3 == labels_t).sum().item()
            acc_adv_R_correct[4] += (preds_R_adv4 == labels_t).sum().item()
            acc_adv_R_correct[5] += (preds_R_adv5 == labels_t).sum().item()
            acc_adv_R_correct[6] += (preds_R_adv6 == labels_t).sum().item()
            acc_adv_R_correct[7] += (preds_R_adv7 == labels_t).sum().item()
            acc_adv_R_correct[8] += (preds_R_adv8 == labels_t).sum().item()
            acc_adv_R_correct[9] += (preds_R_adv9 == labels_t).sum().item()

            acc_adv_f_correct[0] += (preds_f_adv0 == labels_t).sum().item()
            acc_adv_f_correct[1] += (preds_f_adv1 == labels_t).sum().item()
            acc_adv_f_correct[2] += (preds_f_adv2 == labels_t).sum().item()
            acc_adv_f_correct[3] += (preds_f_adv3 == labels_t).sum().item()
            acc_adv_f_correct[4] += (preds_f_adv4 == labels_t).sum().item()
            acc_adv_f_correct[5] += (preds_f_adv5 == labels_t).sum().item()
            acc_adv_f_correct[6] += (preds_f_adv6 == labels_t).sum().item()
            acc_adv_f_correct[7] += (preds_f_adv7 == labels_t).sum().item()
            acc_adv_f_correct[8] += (preds_f_adv8 == labels_t).sum().item()
            acc_adv_f_correct[9] += (preds_f_adv9 == labels_t).sum().item()

    # convert to percentages
    clean_R_acc = 100.0 * correct_clean_R / total_test
    clean_f_acc = 100.0 * correct_clean_f / total_test

    adv_accs_R_correct = [100.0 * acc_adv_R_correct[i] / total_test for i in range(10)]
    adv_accs_f_correct = [100.0 * acc_adv_f_correct[i] / total_test for i in range(10)]
    mean_adv_R_correct = sum(adv_accs_R_correct) / 10.0
    mean_adv_f_correct = sum(adv_accs_f_correct) / 10.0

    print(f" Eval - clean: R {clean_R_acc:.2f}% | f {clean_f_acc:.2f}%")
    for i in range(10):
        print(f"  Adv (G{i} target={i}) - R correct%: {adv_accs_R_correct[i]:.2f}% | f correct%: {adv_accs_f_correct[i]:.2f}%")
    print(f" Mean adv-correct (R): {mean_adv_R_correct:.2f}% | Mean adv-correct (f): {mean_adv_f_correct:.2f}%")

    # step scheduler for R_net (once per epoch)
    scheduler_R.step()

    t1 = time.time()
    print(f" Epoch time: {t1 - t0:.1f}s\n")




In [ ]:
# ===============================================
# In the three cells below:
# T-net is trined using Model ResNet-18.
# To train T-net with Model ResNet-32 or Model Wide ResNet-34, as mentioned in the R-GAN Paper, set T-net according to the architecture of these models.
# The Architecture of Model ResNet-32 or Wide ResNet-34 is defined in the "Architecture of Models" file.
# ===============================================

In [ ]:
# Shuffle the train dataset

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
print("The Train Dataset is shuffled in order difference with Victim Model")

In [ ]:
# ===============================================
# In This Cell:
# The Architecture of ResNet-18 (T-net) is defined (CIFAR-10 version).
# ===============================================


# Basic residual block
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ResNet-18 adapted for CIFAR-10
class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_CIFAR, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)  # global avg pool
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18_CIFAR():
    return ResNet_CIFAR(BasicBlock, [2,2,2,2])


In [ ]:

# ===============================================
# In This Cell:
# The classifier of T-net is trained.
# ===============================================

import torch
import torch.nn as nn
import torch.optim as optim

# build T_net
T_net = ResNet18_CIFAR().to(device)

# loss / optimizer / scheduler 
criterion_t = nn.CrossEntropyLoss()
optimizer_t = optim.SGD(T_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_t = optim.lr_scheduler.MultiStepLR(optimizer_t, milestones=[80, 120], gamma=0.1)

epochs_t = 150
print("Training T_net (ResNet18) for", epochs_t, "epochs...")

for epoch in range(1, epochs_t + 1):
    # ---- Train ----
    T_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)

        optimizer_t.zero_grad()
        logits = T_net(imgs_norm)           # imgs_norm assumed normalized by the transforms
        loss = criterion_t(logits, labels)
        loss.backward()
        optimizer_t.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    train_loss = running_loss / (len(train_dataloader) if len(train_dataloader)>0 else 1)

    # ---- Test / eval periodically ----
    if (epoch == 1 or epoch % 10 == 0 or epoch == epochs_t):
        T_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm = imgs_t_norm.to(device)
                labels_t = labels_t.to(device)
                logits_t = T_net(imgs_t_norm)
                loss_t = criterion_t(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)

        test_acc = 100.0 * correct_test / total_test if total_test>0 else 0.0
        test_loss = test_loss / (len(test_dataloader) if len(test_dataloader)>0 else 1)
        print(f"[T_train] Epoch {epoch}/{epochs_t} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

    scheduler_t.step()

# Freeze T_net and set eval (like that did for f_net)
T_net.eval()
for p in T_net.parameters():
    p.requires_grad = False

print("Finished training T_net; model set to eval and frozen.")


In [ ]:

# ================================================
# In This Cell: 
# The accuracy of T_net, R_net, and f_net for perturbed data by generators G0..G9 and clean data is evaluated.
# In this file, the generator G_i produces Targeted Attacks towards label "i" (e.g., target G9 = class 9).
# ================================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

# Ensure all models in eval mode
f_net.eval()
R_net.eval()
T_net.eval()

G_list = [G0, G1, G2, G3, G4, G5, G6, G7, G8, G9]   # all 10 generators
num_G = len(G_list)

try:
    eps
except NameError:
    eps = 8.0 / 255.0

# --- Clean accuracy counters ---
total = 0
correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

# --- Adversarial accuracy counters per generator ---
correct_adv_f = [0] * num_G
correct_adv_R = [0] * num_G
correct_adv_T = [0] * num_G

print("Running evaluation on clean and each G_i adversaries ...")

with torch.no_grad():
    for imgs_norm, labels in tqdm(test_dataloader):
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)
        B = labels.size(0)
        total += B

        # ========== CLEAN ACCURACY (once for all generators) ==========
        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        preds_T_clean = T_net(imgs_norm).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

        # convert clean to pixel space
        imgs_pixel = denormalize(imgs_norm)

        # ========== LOOP OVER EACH GENERATOR G_i ==========
        for i, G in enumerate(G_list):

            delta = G(imgs_pixel)
            delta = torch.clamp(delta, -eps, eps)

            adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
            adv_norm = renormalize(adv_pixel)

            # ---- f_net ----
            preds_f_adv = f_net(adv_norm).argmax(dim=1)
            correct_adv_f[i] += (preds_f_adv == labels).sum().item()

            # ---- R_net ----
            preds_R_adv = R_net(adv_norm).argmax(dim=1)
            correct_adv_R[i] += (preds_R_adv == labels).sum().item()

            # ---- T_net ----
            preds_T_adv = T_net(adv_norm).argmax(dim=1)
            correct_adv_T[i] += (preds_T_adv == labels).sum().item()

# ========================== RESULTS ===============================

clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

print("\n================ CLEAN ACCURACIES ================")
print(f"f_net clean acc: {clean_acc_f:.2f}%")
print(f"R_net clean acc: {clean_acc_R:.2f}%")
print(f"T_net clean acc: {clean_acc_T:.2f}%")

print("\n================ ADVERSARIAL ACCURACIES PER GENERATOR ================")
for i in range(num_G):
    adv_acc_f = 100.0 * correct_adv_f[i] / total
    adv_acc_R = 100.0 * correct_adv_R[i] / total
    adv_acc_T = 100.0 * correct_adv_T[i] / total

    print(f"\n--- Generator G{i} (target={i}) ---")
    print(f"f_net adv acc: {adv_acc_f:.2f}%   (drop {clean_acc_f - adv_acc_f:.2f} pp)")
    print(f"R_net adv acc: {adv_acc_R:.2f}%   (drop {clean_acc_R - adv_acc_R:.2f} pp)")
    print(f"T_net adv acc: {adv_acc_T:.2f}%   (drop {clean_acc_T - adv_acc_T:.2f} pp)")
